In [3]:
import tensorflow as tf
import cv2
import sklearn

print(tf.__version__)
print(cv2.__version__)

2.21.0
5.0.0


In [5]:
import os
import numpy as np
import cv2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

data_dir = "images"   
img_size = 64         

X = []
y = []

print(" Loading dataset from:", data_dir)

for idx, folder in enumerate(os.listdir(data_dir)):
    folder_path = os.path.join(data_dir, folder)
    if not os.path.isdir(folder_path):
        continue  

    print(f" Processing folder {folder} (class {idx})")

    for file in os.listdir(folder_path):
        img_path = os.path.join(folder_path, file)
        try:
            img = cv2.imread(img_path)
            if img is None:
                print("    Skipping (not an image):", img_path)
                continue
            img = cv2.resize(img, (img_size, img_size))
            X.append(img)
            y.append(idx)
        except Exception as e:
            print("    Error reading:", img_path, "Error:", e)

print(" Finished loading images")
print("Total samples:", len(X))

X = np.array(X, dtype="float32") / 255.0   
y = to_categorical(np.array(y))            

print(" Dataset prepared")
print("X shape:", X.shape)
print("y shape:", y.shape)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(" Train-test split done")
print("Training samples:", X_train.shape[0], "Testing samples:", X_test.shape[0])

print(" Building CNN model...")

model = Sequential([
    Input(shape=(img_size, img_size, 3)),          
    Conv2D(32, (3,3), activation='relu'),
    MaxPooling2D(pool_size=(2,2)),
    
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(pool_size=(2,2)),
    
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(y.shape[1], activation='softmax')        
])

print(" Model built successfully")
model.summary()   

print( "Compiling model...")
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])
print(" Model compiled")

print(" Starting training...")
history = model.fit(X_train, y_train,
                    validation_data=(X_test, y_test),
                    epochs=10,
                    batch_size=32,
                    verbose=1)
print(" Training complete")

print(" Evaluating model...")
loss, acc = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {acc:.2f}")


 Loading dataset from: images
 Processing folder 1-palm (class 0)
 Processing folder 10-down (class 1)
 Processing folder 11-none (class 2)
 Processing folder 2-L (class 3)
 Processing folder 3-fist (class 4)
 Processing folder 4-moved_fist (class 5)
 Processing folder 5-thumb (class 6)
 Processing folder 6-index (class 7)
 Processing folder 7-ok (class 8)
 Processing folder 8-moved_palm (class 9)
 Processing folder 9-c (class 10)
 Finished loading images
Total samples: 3300
 Dataset prepared
X shape: (3300, 64, 64, 3)
y shape: (3300, 11)
 Train-test split done
Training samples: 2640 Testing samples: 660
 Building CNN model...
 Model built successfully


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 62, 62, 32)          │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 31, 31, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 29, 29, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 14, 14, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 12544)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │       1,605,760 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 11)                  │           1,419 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,626,571 (6.20 MB)

 Trainable params: 1,626,571 (6.20 MB)

 Non-trainable params: 0 (0.00 B)

Compiling model...
 Model compiled
 Starting training...
Epoch 1/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 10s 93ms/step - accuracy: 0.4258 - loss: 1.7220 - val_accuracy: 0.8606 - val_loss: 0.6116
Epoch 2/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 7s 87ms/step - accuracy: 0.8564 - loss: 0.4821 - val_accuracy: 0.9697 - val_loss: 0.1486
Epoch 3/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 5s 54ms/step - accuracy: 0.9371 - loss: 0.2123 - val_accuracy: 0.9909 - val_loss: 0.0707
Epoch 4/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 4s 51ms/step - accuracy: 0.9659 - loss: 0.1149 - val_accuracy: 0.9894 - val_loss: 0.0333
Epoch 5/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 6s 72ms/step - accuracy: 0.9814 - loss: 0.0685 - val_accuracy: 0.9985 - val_loss: 0.0114
Epoch 6/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 7s 84ms/step - accuracy: 0.9883 - loss: 0.0463 - val_accuracy: 1.0000 - val_loss: 0.0088
Epoch 7/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 5s 58ms/step - accuracy: 0.9928 - loss: 0.0338 - val_accuracy: 1.0000 - val_loss: 0.0027
Epoch 8/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 5s 55ms/step - 

In [6]:
import os
import numpy as np
import cv2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

In [7]:
data_dir = "images"   
img_size = 64         

X = []
y = []

print(" Loading dataset from:", data_dir)

for idx, folder in enumerate(os.listdir(data_dir)):
    folder_path = os.path.join(data_dir, folder)
    if not os.path.isdir(folder_path):
        continue  

    print(f" Processing folder {folder} (class {idx})")

    for file in os.listdir(folder_path):
        img_path = os.path.join(folder_path, file)
        try:
            img = cv2.imread(img_path)
            if img is None:
                print("    Skipping (not an image):", img_path)
                continue
            img = cv2.resize(img, (img_size, img_size))
            X.append(img)
            y.append(idx)
        except Exception as e:
            print("    Error reading:", img_path, "Error:", e)

print(" Finished loading images")

 Loading dataset from: images
 Processing folder 1-palm (class 0)
 Processing folder 10-down (class 1)
 Processing folder 11-none (class 2)
 Processing folder 2-L (class 3)
 Processing folder 3-fist (class 4)
 Processing folder 4-moved_fist (class 5)
 Processing folder 5-thumb (class 6)
 Processing folder 6-index (class 7)
 Processing folder 7-ok (class 8)
 Processing folder 8-moved_palm (class 9)
 Processing folder 9-c (class 10)
 Finished loading images
